In [ ]:
# %%
# plot_simulations.py (Jupyter-compatible)
import json

import matplotlib.pyplot as plt
import numpy as np

# Inline plotting if run in Jupyter
%matplotlib inline

def plot_simulation_results(json_file="simulation_results.json", dim=None, output_dim=None):
    """
    Plot best outputs (default) or a selected dimension of best_input across simulations.

    Args:
        json_file (str): Path to the JSON results file.
        dim (int or None): If None, plots best_output; if int, plots that dimension of best_input.
        output_dim (int or None): If dim is None, select which dimension of best_output to plot.
    """
    # Load JSON data
    with open(json_file) as f:
        data = json.load(f)

    if not data:
        print("No simulations found in JSON file.")
        return

    # Sort simulations by their sim_xx key
    sims = sorted(data.keys())

    # Extract common info
    jobnames = [data[k]["jobname"] for k in sims]
    targets = [data[k].get("target", None) for k in sims]

    # --- Determine what to plot ---
    if dim is None:
        # Plot best_output
        all_outputs = [np.array(data[k]["best_output"], dtype=float) for k in sims]
        max_len = max(len(o) for o in all_outputs)

        # If output_dim not specified, default to first dimension
        odim = 0 if output_dim is None else output_dim
        if odim >= max_len:
            print(f"Requested output_dim={odim} exceeds available dimensions ({max_len}).")
            return

        values = np.array([o[odim] if odim < len(o) else np.nan for o in all_outputs], dtype=float)
        ylabel = f"Best Output [Dimension {odim}]"
        title = f"Best Output Dimension {odim} per Simulation"
    else:
        # Plot best_input
        values = []
        for k in sims:
            best_input = np.array(data[k]["best_input"], dtype=float).flatten()
            if dim < len(best_input):
                values.append(best_input[dim])
            else:
                values.append(np.nan)
        values = np.array(values, dtype=float)
        ylabel = f"Best Input [Dimension {dim}]"
        title = f"Best Input Dimension {dim} per Simulation"

    # --- Identify global best ---
    if not np.isnan(values).all():
        best_idx = np.nanargmax(values)
        best_value = values[best_idx]
    else:
        print("No valid data to plot.")
        return

    # --- Plot ---
    plt.figure(figsize=(9, 5))
    plt.plot(np.arange(len(sims)), values, marker="o", lw=1.8, label=ylabel)
    plt.scatter(best_idx, best_value, color="red", s=80, zorder=5,
                label=f"Global Best: {sims[best_idx]} = {best_value:.3f}")

    # Annotate with jobname/target
    for i, (job, tgt) in enumerate(zip(jobnames, targets)):
        label = f"{job}"
        if tgt is not None:
            # If target is multi-dimensional, convert to string
            tgt_str = (
                np.round(tgt, 2).tolist() if isinstance(tgt, (list, np.ndarray)) else np.round(tgt, 2)
            )
            label += f" | Target={tgt_str}"
        plt.text(i, values[i], label, fontsize=8, ha="center", va="bottom", rotation=25)

    plt.title(title, fontsize=14)
    plt.xlabel("Simulation Index", fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()



In [ ]:
# Example usage in Jupyter:
# plot all
json_file = "/d/gitlab/ensima-code/optimization/SeatShell.json"
plot_simulation_results(json_file)


plot only one of the dimensions

In [ ]:
json_file = "/d/gitlab/ensima-code/optimization/SeatShell.json"
plot_simulation_results(json_file, dim=1)